In [30]:
import yfinance as yf
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lag, round as spark_round
from pyspark.sql.window import Window
import os

# 環境変数設定（重要: WindowsのSparkでParquet書き込み対策）
try:
    import env_setup
except ImportError:
    pass  # なければ無視

In [31]:
spark = SparkSession.builder.appName("Load Parquet").master("local[*]").getOrCreate()

In [32]:
spark_df = spark.read.parquet("../data/processed/stock_prices.parquet")
spark_df.show(2)

+-------------------+------+-----------------+-----------------+-----------------+-----------------+-----------------+--------+----------+---------+
|               Date|Ticker|             Open|             High|              Low|            Close|        Adj Close|  Volume|Prev_Close|Return(%)|
+-------------------+------+-----------------+-----------------+-----------------+-----------------+-----------------+--------+----------+---------+
|2018-01-02 00:00:00|  BP.L|524.2000122070312|524.2999877929688|514.2999877929688|            517.5|515.2674560546875|20152548|      null|     null|
|2018-01-03 00:00:00|  BP.L|519.2000122070312|525.5999755859375|518.7999877929688|524.2000122070312|521.9384155273438|27207749|     517.5|     1.29|
+-------------------+------+-----------------+-----------------+-----------------+-----------------+-----------------+--------+----------+---------+
only showing top 2 rows



In [33]:
df_pd = spark_df.toPandas()
df_pd.head()

c:\Users\Pupi\Desktop\Git project\stock-trend-etl-spark\.venv\lib\site-packages\pyspark\sql\pandas\conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)


,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,Prev_Close,Return(%)
0,2018-01-02,BP.L,524.200012,524.299988,514.299988,517.500000,515.267456,20152548,NaN,NaN
1,2018-01-03,BP.L,519.200012,525.599976,518.799988,524.200012,521.938416,27207749,517.500000,1.29
2,2018-01-04,BP.L,529.099976,530.599976,527.200012,530.000000,527.713562,32592520,524.200012,1.11
3,2018-01-05,BP.L,530.000000,531.299988,526.700012,529.599976,527.315308,24779506,530.000000,-0.08
4,2018-01-08,BP.L,529.400024,530.299988,526.900024,527.400024,525.124695,29326873,529.599976,-0.42


In [34]:
 
df_pd =df_pd.sort_values(['Ticker', 'Date'])
## Prev_Close
df_pd["Prev_Close"] = df_pd.groupby(["Ticker"])["Close"].shift(1)
## Return
df_pd["Return"] = (df_pd["Close"] - df_pd["Prev_Close"] / df_pd["Prev_Close"]) * 100

In [35]:
## Normalised Close
df_pd["Normalised Close"] = df_pd.groupby(["Ticker"])["Close"].transform(lambda x: x/x.iloc[0])

In [39]:
Rolling_Window = 30

## AvgReturn and std over 30 days window
mean_r = df_pd.groupby(["Ticker"])["Return"].rolling(Rolling_Window).mean()
std_r = df_pd.groupby(["Ticker"])["Return"].rolling(Rolling_Window).std()

sharp = mean_r / std_r
sharp

Ticker      
BP.L    0             NaN
        1             NaN
        2             NaN
        3             NaN
        4             NaN
                  ...    
VOD.L   5296    41.848665
        5297    41.779082
        5298    42.707192
        5299    41.354245
        5300    39.990629
Name: Return, Length: 5301, dtype: float64

In [ ]:
## 